# 2 - Data Pipeline

In [1]:
import pandas as pd

In [2]:
raw_data_fp = "../data/raw"
sales_train_validation = pd.read_csv(f"{raw_data_fp}/sales_train_validation.csv")
calendar = pd.read_csv(f"{raw_data_fp}/calendar.csv")
sell_prices = pd.read_csv(f"{raw_data_fp}/sell_prices.csv")

## Step 1: Plan the Transformation

The raw M5 tables will be transformed into an `item_store_day` dataset at one item-store-day grain. `sales_train_validation` will be unpivoted using Pandas' `melt()` function to create daily sales observations. The resulting data will be joined to `calendar` using `d` to add the corresponding date and `wm_yr_wk`. It will then be joined to `sell_prices` using `item_id`, `store_id`, and `wm_yr_wk` to add the applicable price. Finally, `daily_revenue` will be calculated as `units_sold * sell_price`.

In [3]:
sales_train_validation.head(3)

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1


In [4]:
cols_to_unpivot = sales_train_validation.columns[
    sales_train_validation.columns.str.startswith("d_")
]
item_store_day = pd.melt(
    sales_train_validation,
    id_vars=["item_id", "store_id"],
    value_vars=cols_to_unpivot,
    var_name="d",
    value_name="units_sold"
)

In [5]:
item_store_day.shape

(58327370, 4)

In [6]:
all(item_store_day.columns == ["item_id", "store_id", "d", "units_sold"])

True

In [7]:
# Is the primary key unique in item_store_day?
pk_list = ["item_id", "store_id", "d"]
print(len(item_store_day[pk_list]))
print(len(item_store_day[pk_list].drop_duplicates()))
print(
    len(item_store_day[pk_list]) == 
    len(item_store_day[pk_list].drop_duplicates())
)
# ^ True, so, yes, the primary key is unique

58327370
58327370
True


In [8]:
item_store_day.head()

,item_id,store_id,d,units_sold
0,HOBBIES_1_001,CA_1,d_1,0
1,HOBBIES_1_002,CA_1,d_1,0
2,HOBBIES_1_003,CA_1,d_1,0
3,HOBBIES_1_004,CA_1,d_1,0
4,HOBBIES_1_005,CA_1,d_1,0


In [9]:
item_store_day = pd.merge(
    item_store_day,
    calendar[["d", "date", "wm_yr_wk"]],
    how="left",
    on=["d"],
    validate="many_to_one"
)

In [10]:
# Is the primary key STILL  unique in item_store_day?
pk_list = ["item_id", "store_id", "d"]
isd_pk_df = item_store_day[pk_list].copy()
len_isd_pk_df = len(isd_pk_df)
len_isd_pk_df_drop_dups = len(isd_pk_df.drop_duplicates())
check = (len_isd_pk_df == len_isd_pk_df_drop_dups)
print(f"Row Count of item_store_day: \n{len_isd_pk_df}")
print(f"Row Count of item_store_day (w/ duplicate rows dropped): \n{len_isd_pk_df_drop_dups}")
if check:
    print("\nTherefore, YES, the primary key of item_store_day is unique\n")
else:
    print("\nTherefore, NO, the primary key of item_store_day is NOT unique\n")

Row Count of item_store_day: 
58327370
Row Count of item_store_day (w/ duplicate rows dropped): 
58327370

Therefore, YES, the primary key of item_store_day is unique



In [20]:
item_store_day.head()

,item_id,store_id,d,units_sold,date,wm_yr_wk
0,HOBBIES_1_001,CA_1,d_1,0,2011-01-29,11101
1,HOBBIES_1_002,CA_1,d_1,0,2011-01-29,11101
2,HOBBIES_1_003,CA_1,d_1,0,2011-01-29,11101
3,HOBBIES_1_004,CA_1,d_1,0,2011-01-29,11101
4,HOBBIES_1_005,CA_1,d_1,0,2011-01-29,11101


In [21]:
sell_prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [30]:
item_store_day = pd.merge(
    item_store_day,
    sell_prices,
    how="left",
    on=["item_id", "store_id", "wm_yr_wk"],
    validate="many_to_one"
)

In [32]:
item_store_day["daily_revenue"] = item_store_day["units_sold"] * item_store_day["sell_price"]

### Various validation checks below

In [35]:
item_store_day.shape

(58327370, 8)

In [38]:
pk_list = ["item_id", "store_id", "d"]
len(item_store_day[pk_list]) == len(item_store_day[pk_list].drop_duplicates())

True

In [40]:
expected_isd_cols = {
    "item_id", 
    "store_id", 
    "d", 
    "date", 
    "wm_yr_wk", 
    "units_sold", 
    "sell_price", 
    "daily_revenue"
}
set(item_store_day.columns) == expected_isd_cols

True

In [45]:
sum(item_store_day["sell_price"].isna())

12299413

In [53]:
item_store_day.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 8 columns):
 #   Column         Non-Null Count     Dtype  
---  ------         --------------     -----  
 0   item_id        58327370 non-null  object 
 1   store_id       58327370 non-null  object 
 2   d              58327370 non-null  object 
 3   units_sold     58327370 non-null  int64  
 4   date           58327370 non-null  object 
 5   wm_yr_wk       58327370 non-null  int64  
 6   sell_price     46027957 non-null  float64
 7   daily_revenue  46027957 non-null  float64
dtypes: float64(2), int64(2), object(4)
memory usage: 3.5+ GB


In [56]:
len(item_store_day) - 46027957

12299413

In [ ]:
item_store_day_sp_notna_df = item_store_day[item_store_day["sell_price"].notna()]
all(
    item_store_day_sp_notna_df["daily_revenue"] == 
    (item_store_day_sp_notna_df["units_sold"] * item_store_day_sp_notna_df["sell_price"])
)

True

In [61]:
del item_store_day_sp_notna_df

## Step 2: Determine Pandas dtypes

In [65]:
item_store_day.head(3)

,item_id,store_id,d,units_sold,date,wm_yr_wk,sell_price,daily_revenue
0,HOBBIES_1_001,CA_1,d_1,0,2011-01-29,11101,NaN,NaN
1,HOBBIES_1_002,CA_1,d_1,0,2011-01-29,11101,NaN,NaN
2,HOBBIES_1_003,CA_1,d_1,0,2011-01-29,11101,NaN,NaN


In [66]:
item_store_day.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 8 columns):
 #   Column         Dtype  
---  ------         -----  
 0   item_id        object 
 1   store_id       object 
 2   d              object 
 3   units_sold     int64  
 4   date           object 
 5   wm_yr_wk       int64  
 6   sell_price     float64
 7   daily_revenue  float64
dtypes: float64(2), int64(2), object(4)
memory usage: 3.5+ GB


In [69]:
item_store_day["d"] = item_store_day['d'].str[2:].astype("int")

In [70]:
item_store_day.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 8 columns):
 #   Column         Dtype  
---  ------         -----  
 0   item_id        object 
 1   store_id       object 
 2   d              int64  
 3   units_sold     int64  
 4   date           object 
 5   wm_yr_wk       int64  
 6   sell_price     float64
 7   daily_revenue  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 3.5+ GB


In [72]:
item_store_day["date"] = pd.to_datetime(item_store_day["date"])

In [73]:
item_store_day.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   item_id        object        
 1   store_id       object        
 2   d              int64         
 3   units_sold     int64         
 4   date           datetime64[ns]
 5   wm_yr_wk       int64         
 6   sell_price     float64       
 7   daily_revenue  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(2)
memory usage: 3.5+ GB


In [74]:
isd_cols = [
    "item_id", 
    "store_id", 
    "d", 
    "date", 
    "wm_yr_wk", 
    "units_sold", 
    "sell_price", 
    "daily_revenue"
]
item_store_day = item_store_day[isd_cols]

In [76]:
item_store_day.shape

(58327370, 8)

In [77]:
item_store_day.head()

,item_id,store_id,d,date,wm_yr_wk,units_sold,sell_price,daily_revenue
0,HOBBIES_1_001,CA_1,1,2011-01-29,11101,0,NaN,NaN
1,HOBBIES_1_002,CA_1,1,2011-01-29,11101,0,NaN,NaN
2,HOBBIES_1_003,CA_1,1,2011-01-29,11101,0,NaN,NaN
3,HOBBIES_1_004,CA_1,1,2011-01-29,11101,0,NaN,NaN
4,HOBBIES_1_005,CA_1,1,2011-01-29,11101,0,NaN,NaN


# Step 3: Persist the transformed dataset

In [84]:
# item_store_day.to_parquet("../data/processed/item_store_day.parquet")
# note: this did not work initially

In [80]:
import pandas as pd
import pyarrow as pa

print(pd.__version__)
print(pa.__version__)

2.3.3
25.0.1


In [81]:
import pyarrow.parquet as pq

In [82]:
table = pa.Table.from_pandas(item_store_day)
pq.write_table(table, "../data/processed/item_store_day.parquet")

### Verify Parquet file was successfully created

In [89]:
# pq_verify_df = pd.read_parquet("../data/processed/item_store_day.parquet")
# note: this didn't work

In [86]:
table = pq.read_table("../data/processed/item_store_day.parquet")

In [88]:
table.schema

item_id: string
store_id: string
d: int64
date: timestamp[ns]
wm_yr_wk: int64
units_sold: int64
sell_price: double
daily_revenue: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1208